# Two-Stage Residual Model for NFL Draft Signals

Builds a baseline logistic rating model, captures the surprising successes as residuals, and trains a pillar-aware regressor to explain that signal.

## Overview

- **Stage 1:** Logistic regression on `rating` only to capture the expected chance of landing a second contract (`expected_success`).
- **Stage 2:** Random Forest regressor on the residual `success_residual = made_it_contract - expected_success` using the four pillar scores from the NFL-Way preprocessing pipeline.
- Compare the residual model by feature importance and surface the *Top 10 Hidden Gems* whose actual performance exceeds the rating-based expectation.

In [1]:
import re
import warnings
from itertools import chain

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures
from nltk import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, r2_score
from sklearn.model_selection import cross_val_predict, KFold

In [2]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sffra\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sffra\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sffra\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\sffra\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## Controls & Constants

In [5]:
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
DATA_PATH = 'data/processed/draft_enriched_with_contracts.csv'
TARGET = 'made_it_contract'
PILLAR_COLS = [
    'score_athletic',
    'score_technical',
    'score_character',
    'score_iq',
]
STRAT_LEVEL = 'Pos_Group'
YEAR_MIN = 2014
YEAR_MAX = 2021
GRADE_MAX = 8.5
RF_REG_PARAMS = dict(
    n_estimators=250,
    max_depth=12,
    min_samples_leaf=4,
    random_state=RANDOM_STATE,
)
CV_FOLDS = 5


## Data Load & Filtering

In [7]:
df = pd.read_csv('../data/processed/draft_enriched_with_contracts.csv')
df = df[(df['year'] >= YEAR_MIN) & (df['year'] <= YEAR_MAX)].copy()
df = df[df['grade'] <= GRADE_MAX].copy()
df = df.dropna(subset=[TARGET, 'grade', 'Pos_Group', 'position']).copy()
df['rating'] = df['grade']
df[TARGET] = df[TARGET].astype(int)
df = df.reset_index(drop=True)
print(f'Loaded {len(df)} prospects with valid grades and contracts.')

Loaded 3372 prospects with valid grades and contracts.


## NFL-Way Strengths Text Preprocessing

In [8]:
for col in ['strengths']:
    df[col] = df[col].fillna('').astype(str)

def normalize_hyphen(text: str) -> str:
    return re.sub(r'[-–—]', ' ', text)

curated_terms = [
    'pad level', 'change of direction', 'press coverage', 'ball skills', 'hand placement',
    'play recognition', 'run blocking', 'anchor strength', 'coverage awareness', 'stack and shed',
    'man coverage', 'zone coverage', 'offensive line', 'route depth', 'field vision',
    'lower body', 'upper body', 'catch radius', 'short area burst', 'long speed',
    'process speed', 'third down', 'chain mover', 'mental processing', 'football iq'
]
curated_map = {term: term.replace(' ', '_') for term in curated_terms}

stop_fillers = {'prospect', 'player', 'shows', 'ability', 'potential'}
stop_words = set(stopwords.words('english'))
preserve_directionals = {'high', 'low', 'deep', 'heavy'}
stop_words -= preserve_directionals
stop_words |= stop_fillers
lemmatizer = WordNetLemmatizer()


def stitch_text(text: str) -> str:
    text = normalize_hyphen(text).lower()
    for phrase, repl in curated_map.items():
        text = re.sub(rf'{re.escape(phrase)}', repl, text)
    return text

df['strengths_curated'] = df['strengths'].apply(stitch_text)

all_tokens = list(chain.from_iterable(word_tokenize(doc) for doc in df['strengths_curated']))
finder = BigramCollocationFinder.from_words(all_tokens)
finder.apply_freq_filter(2)
measures = BigramAssocMeasures()
top_bigrams = finder.nbest(measures.pmi, 30)
bigram_map = {' '.join(bg): '_'.join(bg) for bg in top_bigrams}

def stitch_bigrams(text: str) -> str:
    for phrase, repl in bigram_map.items():
        text = re.sub(rf'{re.escape(phrase)}', repl, text)
    return text

df['strengths_bigrams'] = df['strengths_curated'].apply(stitch_bigrams)


def is_alpha_like(token: str) -> bool:
    return token.replace('_', '').isalpha()

def tokenize_and_lemmatize(text: str) -> list:
    tokens = []
    for token in word_tokenize(text):
        token_lower = token.lower()
        if token_lower in stop_words or not is_alpha_like(token_lower):
            continue
        lemma = lemmatizer.lemmatize(token_lower.replace('_', ' '))
        tokens.append(lemma.replace(' ', '_'))
    return tokens

df['tokens'] = df['strengths_bigrams'].apply(tokenize_and_lemmatize)
df['clean_strengths'] = df['tokens'].apply(' '.join)
print('Preprocessing complete. Example tokens:')
print(df['tokens'].head(3).tolist())

Preprocessing complete. Example tokens:
[['look', 'every', 'bit', 'part', 'athletic', 'marvel', 'raw', 'explosive', 'power', 'rare', 'speed', 'size', 'physically', 'tough', 'battle', 'injury', 'collapse', 'corner', 'ease', 'rag', 'doll', 'blocker', 'highly', 'disruptive', 'creates', 'lot', 'pressure', 'flush', 'production', 'teammate', 'split', 'double', 'team', 'close', 'hiccup', 'play', 'leverage', 'power', 'hand', 'convert', 'speed', 'power', 'bull', 'blocker', 'backfield', 'disrupts', 'quarterback', 'vision', 'long', 'arm', 'bat', 'ball', 'seldom', 'leaf', 'field', 'flash', 'playmaking', 'produce', 'athletic', 'feat', 'category', 'others', 'versatile', 'line', 'everywhere', 'along', 'line', 'win', 'strength', 'power', 'quickness', 'speed', 'personality', 'pleaser', 'like', 'disappoint', 'coach', 'teammate'], ['exceptional', 'football', 'playing', 'speed', 'flat', 'fly', 'take', 'top', 'defense', 'world', 'class', 'track', 'speed', 'extends', 'outside', 'frame', 'pluck', 'ball', 'ou

## Pillar Scoring (TF-IDF + Archetypes)

In [9]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['clean_strengths'])
archetypes = {
    'athletic': 'Elite physical athleticism, speed, explosiveness, and body control.',
    'technical': 'Refined technical mechanics, hand usage, footwork, and positional skill.',
    'character': 'High motor, competitive toughness, leadership, and relentless effort.',
    'iq': 'High football IQ, play recognition, mental processing, and anticipation.',
}
archetype_keys = list(archetypes.keys())
archetype_matrix = vectorizer.transform([archetypes[k] for k in archetype_keys])
similarity_matrix = cosine_similarity(tfidf_matrix, archetype_matrix)
similarity_df = pd.DataFrame(similarity_matrix, columns=archetype_keys, index=df.index)

score_map = {
    'athletic': 'score_athletic',
    'technical': 'score_technical',
    'character': 'score_character',
    'iq': 'score_iq',
}
renamed = similarity_df.rename(columns=score_map)
for col in renamed.columns:
    min_val = renamed[col].min()
    max_val = renamed[col].max()
    if max_val > min_val:
        renamed[col] = 100 * (renamed[col] - min_val) / (max_val - min_val)
    else:
        renamed[col] = 100 * renamed[col]

df = pd.concat([df, renamed], axis=1)
print('Pillar scores added.')
print(df[PILLAR_COLS].head())

Pillar scores added.
   score_athletic  score_technical  score_character   score_iq
0       17.049993         2.563808         0.000000   2.289330
1       29.514726         2.752142         0.000000   8.314634
2        8.739834         0.000000         0.000000  10.561602
3       21.020230         0.000000        11.919512   4.454927
4       16.326749         5.190702         0.000000  10.158441


## Stage 1: Rating-Only Logistic Baseline

In [10]:
X_rating = df[['rating']].copy()
log_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('logit', LogisticRegression(max_iter=500, class_weight='balanced', random_state=RANDOM_STATE)),
])
log_pipe.fit(X_rating, df[TARGET])
df['expected_success'] = log_pipe.predict_proba(X_rating)[:, 1]
log_pr_auc = average_precision_score(df[TARGET], df['expected_success'])
print(f'Baseline logistic PR-AUC (rating only) = {log_pr_auc:.4f}')

Baseline logistic PR-AUC (rating only) = 0.5360


## Stage 2: Residual Random Forest Regressor

In [11]:
df['success_residual'] = df[TARGET] - df['expected_success']
res_cv = cross_val_predict(
    RandomForestRegressor(**RF_REG_PARAMS),
    df[PILLAR_COLS], df['success_residual'],
    cv=KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1
)
res_r2 = r2_score(df['success_residual'], res_cv)
res_model = RandomForestRegressor(**RF_REG_PARAMS)
res_model.fit(df[PILLAR_COLS], df['success_residual'])
df['residual_pred'] = res_model.predict(df[PILLAR_COLS])
print(f'Cross-validated R² for residual model = {res_r2:.3f}')

ValueError: Supported target types are: ('binary', 'multiclass'). Got 'continuous' instead.

## Analysis: Residual Feature Importance

In [ ]:
imp_df = pd.DataFrame({
    'feature': PILLAR_COLS,
    'importance': res_model.feature_importances_,
}).sort_values('importance', ascending=False)
print(imp_df)
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(imp_df['feature'], imp_df['importance'], color='#4C72B0')
ax.set_title('Random Forest Regressor Feature Importances')
ax.set_ylabel('Importance')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Hidden Gems: Actual Success vs. Expectation

In [ ]:
hidden = df[df['success_residual'] > 0].sort_values('success_residual', ascending=False).head(10)
print('Top 10 hidden gems (actual made-it > expected):')
print(hidden[['player_name', 'position', 'rating', 'expected_success', 'success_residual'] + PILLAR_COLS].to_string(index=False))